# Foot Gesture Recognition using Machine Learning

This tutorial demonstrates how to build a machine learning model to recognize different foot gestures.
Foot gesture recognition can be useful in:
- Gaming and entertainment
- Assistive technology for people with disabilities
- Hands-free control systems
- Sports analytics

We'll use accelerometer and gyroscope data to classify different foot movements.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
%matplotlib inline

## Create Synthetic Foot Gesture Data

For this tutorial, we'll create synthetic data representing sensor readings from a foot-mounted device.
In a real-world scenario, this data would come from accelerometers and gyroscopes.

Gestures:
- 0: Rest (no movement)
- 1: Walk
- 2: Run
- 3: Jump
- 4: Kick

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Number of samples per gesture
samples_per_gesture = 200

# Generate synthetic data for each gesture
def generate_gesture_data(gesture_type, n_samples):
    if gesture_type == 'rest':
        # Low movement, low variance
        accel_x = np.random.normal(0, 0.1, n_samples)
        accel_y = np.random.normal(0, 0.1, n_samples)
        accel_z = np.random.normal(9.8, 0.2, n_samples)
        gyro_x = np.random.normal(0, 0.05, n_samples)
        gyro_y = np.random.normal(0, 0.05, n_samples)
        gyro_z = np.random.normal(0, 0.05, n_samples)
        label = 0
    elif gesture_type == 'walk':
        # Moderate periodic movement
        accel_x = np.random.normal(0.5, 1.5, n_samples)
        accel_y = np.random.normal(0.3, 1.2, n_samples)
        accel_z = np.random.normal(9.8, 2.0, n_samples)
        gyro_x = np.random.normal(0, 0.5, n_samples)
        gyro_y = np.random.normal(0, 0.4, n_samples)
        gyro_z = np.random.normal(0, 0.3, n_samples)
        label = 1
    elif gesture_type == 'run':
        # High periodic movement
        accel_x = np.random.normal(1.5, 3.0, n_samples)
        accel_y = np.random.normal(1.0, 2.5, n_samples)
        accel_z = np.random.normal(9.8, 4.0, n_samples)
        gyro_x = np.random.normal(0, 1.2, n_samples)
        gyro_y = np.random.normal(0, 1.0, n_samples)
        gyro_z = np.random.normal(0, 0.8, n_samples)
        label = 2
    elif gesture_type == 'jump':
        # Sudden upward acceleration followed by downward
        accel_x = np.random.normal(0, 2.0, n_samples)
        accel_y = np.random.normal(0, 2.0, n_samples)
        accel_z = np.random.normal(15.0, 5.0, n_samples)
        gyro_x = np.random.normal(0, 0.8, n_samples)
        gyro_y = np.random.normal(0, 0.8, n_samples)
        gyro_z = np.random.normal(0, 0.6, n_samples)
        label = 3
    else:  # kick
        # Forward movement with rotation
        accel_x = np.random.normal(3.0, 2.0, n_samples)
        accel_y = np.random.normal(2.0, 1.5, n_samples)
        accel_z = np.random.normal(9.8, 2.5, n_samples)
        gyro_x = np.random.normal(2.0, 1.0, n_samples)
        gyro_y = np.random.normal(1.5, 0.8, n_samples)
        gyro_z = np.random.normal(1.0, 0.6, n_samples)
        label = 4
    
    return pd.DataFrame({
        'accel_x': accel_x,
        'accel_y': accel_y,
        'accel_z': accel_z,
        'gyro_x': gyro_x,
        'gyro_y': gyro_y,
        'gyro_z': gyro_z,
        'gesture': label
    })

# Generate data for all gestures
gesture_types = ['rest', 'walk', 'run', 'jump', 'kick']
data_frames = [generate_gesture_data(g, samples_per_gesture) for g in gesture_types]

# Combine all data
df = pd.concat(data_frames, ignore_index=True)

# Shuffle the dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"\nGesture distribution:")
print(df['gesture'].value_counts().sort_index())

In [ ]:
# Display first few rows
df.head()

## Data Visualization

In [ ]:
# Create a mapping for gesture names
gesture_names = {0: 'Rest', 1: 'Walk', 2: 'Run', 3: 'Jump', 4: 'Kick'}
df['gesture_name'] = df['gesture'].map(gesture_names)

# Visualize accelerometer data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, axis in enumerate(['accel_x', 'accel_y', 'accel_z']):
    for gesture in range(5):
        gesture_data = df[df['gesture'] == gesture][axis].values[:50]
        axes[idx].plot(gesture_data, label=gesture_names[gesture], alpha=0.7)
    axes[idx].set_title(f'{axis.upper()} (first 50 samples per gesture)')
    axes[idx].set_xlabel('Sample')
    axes[idx].set_ylabel('Acceleration')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize gyroscope data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, axis in enumerate(['gyro_x', 'gyro_y', 'gyro_z']):
    for gesture in range(5):
        gesture_data = df[df['gesture'] == gesture][axis].values[:50]
        axes[idx].plot(gesture_data, label=gesture_names[gesture], alpha=0.7)
    axes[idx].set_title(f'{axis.upper()} (first 50 samples per gesture)')
    axes[idx].set_xlabel('Sample')
    axes[idx].set_ylabel('Angular Velocity')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Feature Engineering

Calculate statistical features from the sensor data

In [ ]:
# Calculate magnitude of acceleration and gyroscope
df['accel_magnitude'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)
df['gyro_magnitude'] = np.sqrt(df['gyro_x']**2 + df['gyro_y']**2 + df['gyro_z']**2)

print("Added magnitude features")
df.head()

## Prepare Data for Training

In [ ]:
# Select features
feature_columns = ['accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y', 'gyro_z', 
                   'accel_magnitude', 'gyro_magnitude']
X = df[feature_columns]
y = df['gesture']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled using StandardScaler")

## Train Random Forest Classifier

In [ ]:
# Create and train the model
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")

## Model Evaluation

In [ ]:
# Make predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")

In [ ]:
# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=[gesture_names[i] for i in range(5)]))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[gesture_names[i] for i in range(5)],
            yticklabels=[gesture_names[i] for i in range(5)])
plt.title('Confusion Matrix - Foot Gesture Recognition')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## Feature Importance

In [ ]:
# Plot feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance for Foot Gesture Recognition')
plt.tight_layout()
plt.show()

print("\nFeature Importance:")
print(feature_importance)

## Making Predictions on New Data

In [ ]:
# Simulate new sensor readings
new_data = pd.DataFrame({
    'accel_x': [0.05, 0.6, 1.8, 0.1, 3.2],
    'accel_y': [0.02, 0.4, 1.2, 0.0, 2.1],
    'accel_z': [9.82, 10.5, 13.0, 16.0, 10.5],
    'gyro_x': [0.01, 0.3, 1.0, 0.5, 2.2],
    'gyro_y': [0.02, 0.2, 0.8, 0.4, 1.6],
    'gyro_z': [0.01, 0.1, 0.5, 0.3, 1.1]
})

# Calculate magnitude features
new_data['accel_magnitude'] = np.sqrt(new_data['accel_x']**2 + new_data['accel_y']**2 + new_data['accel_z']**2)
new_data['gyro_magnitude'] = np.sqrt(new_data['gyro_x']**2 + new_data['gyro_y']**2 + new_data['gyro_z']**2)

# Scale and predict
new_data_scaled = scaler.transform(new_data[feature_columns])
predictions = model.predict(new_data_scaled)

# Display predictions
print("\nPredictions for new data:")
for i, pred in enumerate(predictions):
    print(f"Sample {i+1}: {gesture_names[pred]}")

## Conclusion

In this tutorial, we:
1. Created synthetic foot gesture data with accelerometer and gyroscope readings
2. Visualized the sensor data for different gestures
3. Engineered features including magnitude calculations
4. Trained a Random Forest classifier to recognize 5 different foot gestures
5. Evaluated the model's performance
6. Made predictions on new data

The model achieved high accuracy in distinguishing between different foot gestures.

### Real-World Applications:
- Gaming controllers using foot movements
- Assistive devices for people with limited hand mobility
- Fitness tracking and sports analytics
- Hands-free control in industrial settings

### Next Steps:
- Collect real sensor data from IMU devices
- Implement temporal features (sliding windows)
- Try deep learning models (LSTM, CNN)
- Add more gesture types
- Optimize for real-time prediction